# Cropping procedure test

Loads a couple of DUNE CVN dense images, converts them to sparse `Voxels` via `from_dense()`, applies the `SparseCropper`, and visualises the original image alongside all global and local crops.

In [ ]:
import sys
sys.path.insert(0, "/nfs/data/1/mvicenzi/ml-dune-model")

import torch
import matplotlib.pyplot as plt
import numpy as np

from warpconvnet.geometry.types.voxels import Voxels

from loader.apa_sparse_meta_dataset import APASparseMetaDataset
from dino.cropping import SparseCropper, CropConfig

## Load dataset

In [ ]:
rootdir = "/nfs/data/1/yuhw/cffm-data/prod-jay-100k-truth-2026-02-27"

ds = APASparseMetaDataset(
    rootdir=rootdir,
    apa=0,
    view="W",  
    use_cache=True,
    cache_dir="/nfs/data/1/mvicenzi/ml-dune-model/data",
    return_full_metadata=True,
    )

print("Dataset size:", len(ds))

## Pick two samples and build a batched Voxels

In [ ]:
IDX_A = 1_000
IDX_B = 50_001

x_a, y_a = ds[IDX_A]
x_b, y_b = ds[IDX_B]

for name, x, y in [("A", x_a, y_a), ("B", x_b, y_b)]:
    print(f"Sample {name}: shape={x.feature_tensor.shape} active={int((x.feature_tensor > 0).sum())}\n {y}")

In [ ]:
from loader.collate import voxels_meta_collate_fn
from torch.utils.data import DataLoader

loader = DataLoader(
    ds,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    collate_fn=voxels_meta_collate_fn,
)

vox, _ = next(iter(loader))

print("Voxels batch_size        :", len(vox.offsets) - 1)
print("coordinate_tensor shape  :", vox.coordinate_tensor.shape)
print("feature_tensor shape     :", vox.feature_tensor.shape)
print("offsets                  :", vox.offsets)

## Apply SparseCropper

In [ ]:
cfg = CropConfig(
    n_global=1,
    n_local=2,
    global_scale=(0.5, 0.9),
    local_scale=(0.02, 0.05),
    image_h=500,
    image_w=500,
)

cropper = SparseCropper(cfg)
crops, kept_indices, _ = cropper(vox)

print(f"Number of crops returned : {len(crops)}  (n_global={cfg.n_global}, n_local={cfg.n_local})")
for k, c in enumerate(crops):
    tag = "global" if k < cfg.n_global else "local "
    counts = (c.offsets[1:] - c.offsets[:-1]).tolist()
    print(f"  crop[{k}] ({tag})  active per image: {counts}")

## Plotting helpers

In [ ]:
def scatter_voxels(ax, bic, vals, batch_item, title="", cmap="viridis"):
    """
    Scatter-plot one batch item from a batch_indexed_coordinates array.

    bic  : (N, 3) numpy array — (batch_idx, dim0, dim1)
    vals : (N,) numpy array  — feature values
    """
    mask = bic[:, 0] == batch_item
    # dim0 = row (H), dim1 = col (W)  ->  plot col on x-axis, row on y-axis
    x = bic[mask, 2]   # W (wire / column)
    y = bic[mask, 1]   # H (tick  / row)
    ax.scatter(x, y, c=vals[mask], s=1, cmap=cmap, linewidths=0)
    ax.set_xlim(0, 1050)
    ax.set_ylim(0, 1500)
    ax.invert_yaxis()
    ax.set_xlabel("Wire")
    ax.set_ylabel("Time tick")
    ax.set_title(title, fontsize=9)

## Original images (from sparse Voxels)

In [ ]:
bic_orig  = vox.batch_indexed_coordinates.cpu().numpy()   # (N, 3)
vals_orig = vox.feature_tensor.cpu().numpy()[:, 0]        # (N,)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for b, (ax, idx, label) in enumerate(
    zip(axes, [IDX_A, IDX_B], [y_a["label"], y_b['label']])
):
    scatter_voxels(
        ax, bic_orig, vals_orig, b,
        title=f"Original | sample {idx} | label={label}",
        cmap="twilight",
    )

plt.suptitle("Original APA images (sparse)", fontsize=11)
plt.tight_layout()
plt.show()

## Global crops

In [ ]:
n_images = 2
n_global = cfg.n_global

fig, axes = plt.subplots(n_images, n_global, figsize=(5 * n_global, 5 * n_images), squeeze=False)

for k in range(n_global):
    bic_k  = crops[k].batch_indexed_coordinates.cpu().numpy()
    vals_k = crops[k].feature_tensor.cpu().numpy()[:, 0]
    for b in range(n_images):
        ax = axes[b, k]
        scatter_voxels(
            ax, bic_k, vals_k, b,
            title=f"Global crop {k} | image {b}",
        )

plt.suptitle("Global crops", fontsize=12)
plt.tight_layout()
plt.show()

## Local crops

In [ ]:
n_local  = cfg.n_local

fig, axes = plt.subplots(n_images, n_local, figsize=(5 * n_local, 5 * n_images))

for k_local in range(n_local):
    k = cfg.n_global + k_local   # index into crops list
    bic_k  = crops[k].batch_indexed_coordinates.cpu().numpy()
    vals_k = crops[k].feature_tensor.cpu().numpy()[:, 0]
    for b in range(n_images):
        ax = axes[b, k_local]
        scatter_voxels(
            ax, bic_k, vals_k, b,
            title=f"Local crop {k_local} | image {b}",
        )

plt.suptitle("Local crops", fontsize=12)
plt.tight_layout()
plt.show()

## All crops at a glance — both images side by side

In [ ]:
import matplotlib.gridspec as gridspec

# Layout per image (2 rows × 5 cols):
#   row 0: [original] [global0] [global1] [  empty ] [  empty ]
#   row 1: [ empty  ] [local0 ] [local1 ] [local2  ] [local3  ]

n_rows_per_img = 2
n_cols_gs      = 1 + cfg.n_local   # 1 (orig) + 4 (locals) = 5
total_rows     = n_images * n_rows_per_img   # 4

fig = plt.figure(figsize=(4 * n_cols_gs, 4 * total_rows), dpi=200)
gs  = gridspec.GridSpec(
    total_rows, n_cols_gs,
    figure=fig,
    hspace=0.45,
    wspace=0.30,
)

label_list = [y_a["label"], y_b['label']]

for b in range(n_images):
    r0 = b * n_rows_per_img   # first row for this image
    r1 = r0 + 1               # second row for this image

    # --- Original: row 0, col 0 only ---
    ax_orig = fig.add_subplot(gs[r0, 0])
    scatter_voxels(
        ax_orig, bic_orig, vals_orig, b,
        title=f"Original {b} — {label_list[b]}",
        cmap="twilight",
    )
    # col 0 in row 1 is intentionally left empty

    # --- Global crops: row 0, cols 1 … n_global ---
    for k in range(cfg.n_global):
        ax = fig.add_subplot(gs[r0, 1 + k])
        bic_k  = crops[k].batch_indexed_coordinates.cpu().numpy()
        vals_k = crops[k].feature_tensor.cpu().numpy()[:, 0]
        scatter_voxels(ax, bic_k, vals_k, b, title=f"Global {k}")
    # cols n_global … n_cols_gs-1 in row 0 are intentionally left empty

    # --- Local crops: row 1, cols 1 … n_local ---
    for k_local in range(cfg.n_local):
        ax = fig.add_subplot(gs[r1, 1 + k_local])
        k = cfg.n_global + k_local
        bic_k  = crops[k].batch_indexed_coordinates.cpu().numpy()
        vals_k = crops[k].feature_tensor.cpu().numpy()[:, 0]
        scatter_voxels(ax, bic_k, vals_k, b, title=f"Local {k_local}")

#plt.suptitle("DUNE CVN: original + all crops", fontsize=12, y=1.01)
plt.show()
